# M(EBA) 소수 문항 검증 노트북 (`verify.ipynb`)

담당 B (PR B-5). 채점자 또는 팀원이 **소수 문항만 다시 실행**하여 아래 5가지를 한 번에 확인하는 검증용 노트북입니다 (`EVALUATION §4`, `notebooks/README.md`).

1. **오프라인 단위 테스트** (`pytest`) 통과 여부
2. **동결 데이터(`data/frozen/`) SHA-256 해시** 일치 여부
3. **실모델(`configs/default.yaml`) 검증 실행** (`scripts.run_eval --verify N` → `results/runs/_scratch/` 기록)
4. **채점 재현성 검증**: 생성된 로그를 `src.eval.scorer.score_item` / `is_abstained`로 다시 채점해 차이 **0건** 확인
5. **(정식 로그가 있는 경우)** `scripts.build_tables`로 결과표 생성 확인

## 0. GPU 확인 (`Tesla T4` 확인)

In [ ]:
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

## 1. 검증 설정

In [ ]:
BRANCH = "main"              # 확인할 브랜치명
VERIFY_N = 3                 # 검증 문항 수 (앞 N문항)
ITEM_SET = "eval"            # "eval" (평가 세트 앞 N문항) 또는 "dev" (개발 세트 앞 N문항)
ROUND = "r1"                 # _scratch 아래 라운드 이름
PIPELINES = "p0,p0_raw,p1,m" # 검증할 파이프라인 목록

assert VERIFY_N >= 1
assert ITEM_SET in {"eval", "dev"}

## 2. 코드 받기 (`{BRANCH}`)

In [ ]:
import os, subprocess

os.chdir("/content")
REPO_DIR = "/content/repo"
REPO_URL = "https://github.com/minyi-k03/Intelligence-Software-Project.git"
# 비공개 저장소면 Colab 비밀(userdata)에 GH_TOKEN을 넣고 아래 두 줄 주석을 해제하세요.
# from google.colab import userdata
# REPO_URL = REPO_URL.replace("https://", f"https://{userdata.get('GH_TOKEN')}@")

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run(["git", "fetch", "origin"], check=True)
subprocess.run(["git", "checkout", BRANCH], check=True)
subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], check=True)
!git log --oneline -3

## 3. 의존성 설치 및 오프라인 단위 테스트 (`pytest`)

In [ ]:
!pip install -q -r requirements.txt
!python3 -B -m pytest -q tests

## 4. Ollama 설치·서버 실행 및 모델 GPU 로드 확인

In [ ]:
import subprocess, time, urllib.request, yaml

subprocess.run(["bash", "-lc", "apt-get update -qq && apt-get install -y -qq zstd pciutils lshw > /dev/null"], check=True)
subprocess.run(["bash", "-lc", "which ollama > /dev/null || curl -fsSL https://ollama.com/install.sh | sh"], check=True)

subprocess.run(["bash", "-lc", "pkill -f 'ollama serve' || true"], check=False)
time.sleep(2)

ollama_log = open("/content/ollama.log", "w", encoding="utf-8")
subprocess.Popen(["ollama", "serve"], stdout=ollama_log, stderr=subprocess.STDOUT, start_new_session=True)
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
        print("Ollama 서버 준비됨")
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama 서버가 60초 안에 응답하지 않았습니다.")

tag = yaml.safe_load(open("configs/default.yaml", encoding="utf-8"))["model"]["tag"]
print("model.tag =", tag)
subprocess.run(["ollama", "pull", tag], check=True)
subprocess.run(["ollama", "run", tag, "Say OK."], check=True, timeout=300)
ps = subprocess.run(["ollama", "ps"], capture_output=True, text=True).stdout
print(ps)
assert "GPU" in ps and "CPU" not in ps.replace("PROCESSOR", ""), f"모델이 GPU에 올라가지 않았습니다:\n{ps}"

## 5. 동결 데이터 SHA-256 검증 & `--verify` 실행 (`results/runs/_scratch/`)

In [ ]:
import shutil, subprocess, sys
from pathlib import Path

sys.dont_write_bytecode = True
from src.eval.logger import verify_frozen_hashes

hashes = verify_frozen_hashes(Path("data/frozen"), Path("data/README.md"))
print("동결 해시 일치:", {k: v[:12] for k, v in hashes.items()})

# 이전 _scratch 같은 라운드가 남아 있으면 새로 시작해 설정 충돌을 피한다.
scratch_dir = Path("results/runs/_scratch") / ROUND
if scratch_dir.exists():
    shutil.rmtree(scratch_dir)

cmd = [
    "python3", "-B", "-m", "scripts.run_eval",
    "--verify", str(VERIFY_N),
    "--set", ITEM_SET,
    "--round", ROUND,
    "--pipelines", PIPELINES,
]
print("실행 명령:", " ".join(cmd))
subprocess.run(cmd, check=True)

## 6. 로그 규격 · 채점 재현성 검증 (`score_item` 대조)

In [ ]:
import json
from pathlib import Path
from src.eval.scorer import is_abstained, score_item

scratch_dir = Path("results/runs/_scratch") / ROUND
meta = json.loads((scratch_dir / "RUN_META.json").read_text(encoding="utf-8"))
dataset = {
    json.loads(l)["qid"]: json.loads(l)
    for l in Path(f"data/frozen/{ITEM_SET}.jsonl").read_text(encoding="utf-8").splitlines()
    if l.strip()
}

REQUIRED_KEYS = {
    "qid", "type", "subtype", "answer_kind", "pipeline", "bundle", "round",
    "order_idx", "session_idx", "gold", "answer", "abstained", "budget_exhausted",
    "route", "decisions", "n_calls", "n_retries", "calls", "retrieved_doc_ids",
    "kept_windows", "loop_breaks", "error", "em", "f1", "numeric_correct",
    "correct", "timeout", "latency_s", "max_input_tokens", "evidence_kept", "is_warmup",
}

print("RUN_META pipeline_order:", meta["pipeline_order"], "| VRAM(MiB):", meta.get("max_vram_mib_by_pipeline"))
for name in meta["pipeline_order"]:
    rows = [json.loads(l) for l in (scratch_dir / f"{name}.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]
    assert len(rows) == VERIFY_N, f"{name}: 예상 {VERIFY_N}건, 실제 {len(rows)}건"
    diffs = 0
    for r in rows:
        missing = REQUIRED_KEYS - set(r)
        assert not missing, f"{name}/{r['qid']} 누락 필드: {missing}"
        rescored = score_item(dataset[r["qid"]], r["answer"])
        f1_ok = (
            (rescored["f1"] is None and r["f1"] is None)
            or (rescored["f1"] is not None and r["f1"] is not None and abs(rescored["f1"] - r["f1"]) <= 1e-9)
        )
        if rescored["correct"] != r["correct"] or not f1_ok or is_abstained(r["answer"]) != r["abstained"]:
            diffs += 1
    assert diffs == 0, f"{name}: 재채점 차이 {diffs}건"
    print(f"[{name}] {len(rows)}/{VERIFY_N}건 규격·재채점 일치 확인 (정답 {sum(r['correct'] for r in rows)}/{len(rows)})")
    for r in rows:
        print(f"   {r['qid']} | route={r['route']} | ans={r['answer']!r} | correct={r['correct']} | calls={r['n_calls']} | err={r['error']}")

## 7. 결과표 재현 확인 (정식 `main`·`ablation` 로그가 저장소에 있는 경우)

In [ ]:
import subprocess
from pathlib import Path

has_main = all((Path("results/runs/main") / f"r{i}").is_dir() for i in range(1, 4))
has_ablation = any(Path("results/runs/ablation/r1").glob("m_minus_*.jsonl"))
if has_main and has_ablation:
    out_tmp = "/tmp/verify_tables"
    subprocess.run(["python3", "-B", "-m", "scripts.build_tables", "--output-dir", out_tmp], check=True)
    print("결과표 생성 완료:", [p.name for p in sorted(Path(out_tmp).iterdir())])
else:
    print("아직 정식 본 비교(main r1~r3)·절제(ablation r1) 로그가 없어 결과표 검증은 건너뜁니다.")